# 01. Дизайн эксперимента: персонализированный баннер

Цель — оценить влияние персонализированного баннера на среднее время просмотра после клика. Источник: `data/stream_events.csv`.

**Ограничения исходных данных:** в текущем CSV 100 строк; отсутствуют события `add_to_favorites`, а также признак экспериментальной группы. Поле `watch_time_min` трактуется согласно названию как минуты. Поэтому анализ избранного и фактические результаты A/B-теста по этому файлу получить нельзя. MDE/выборка оцениваются на доступной исторической вариативности и являются предварительными.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
DATA_PATH = ROOT / "data" / "stream_events.csv"
df = pd.read_csv(DATA_PATH)
df["dt"] = pd.to_datetime(df["dt"], errors="coerce")
df["date"] = df["dt"].dt.date
df.head()

## 1. Проверка данных

In [ ]:
print("Размер:", df.shape)
print("Период:", df["dt"].min(), "—", df["dt"].max())
print("Уникальных пользователей:", df["client_id"].nunique())
print("\\nСобытия:")
display(df["event_type"].value_counts(dropna=False).rename_axis("event_type").to_frame("count"))
print("\\nПропуски:")
display(df.isna().sum().rename("missing_count").to_frame())

## 2. Воронка по дням и пользователям

In [ ]:
# Каждая строка — пользователь в конкретный день.
counts = (df.pivot_table(index=["date", "client_id"], columns="event_type",
                         values="dt", aggfunc="count", fill_value=0)
            .reset_index())
counts.columns.name = None
for col in ["view_banner", "click_banner", "start_watch", "finish_watch", "add_to_favorites"]:
    if col not in counts:
        counts[col] = 0

watch = (df[df["event_type"].eq("finish_watch")]
         .groupby(["date", "client_id"])["watch_time_min"].sum()
         .rename("watch_time").reset_index())
funnel = counts.merge(watch, on=["date", "client_id"], how="left")
funnel["watch_time"] = funnel["watch_time"].fillna(0)
funnel = funnel.rename(columns={"date":"dt", "client_id":"user_id",
                                "view_banner":"views", "click_banner":"clicks",
                                "add_to_favorites":"adds_to_fav"})
funnel["ctr"] = np.where(funnel["views"] > 0, funnel["clicks"] / funnel["views"], np.nan)
funnel["avg_watch_time"] = np.where(funnel["clicks"] > 0, funnel["watch_time"] / funnel["clicks"], np.nan)
funnel["cr_fav"] = np.where(funnel["clicks"] > 0, funnel["adds_to_fav"] / funnel["clicks"], np.nan)
funnel["arpu"] = funnel["watch_time"]  # watch time per user-day; агрегируйте по пользователю за период для ARPU
display(funnel.head())
funnel.to_csv(ROOT / "data" / "daily_user_funnel.csv", index=False)

## 3. Метрики и дисперсии

In [ ]:
metric_cols = ["ctr", "avg_watch_time", "cr_fav", "arpu"]
summary = pd.DataFrame({
    "mean": funnel[metric_cols].mean(),
    "variance": funnel[metric_cols].var(ddof=1),
    "std": funnel[metric_cols].std(ddof=1),
    "non_null_n": funnel[metric_cols].count()
})
display(summary)
print("Важно: CR_fav не рассчитывается содержательно, так как в файле нет add_to_favorites; его нулевые значения не следует интерпретировать как реальное поведение.")

## 4. Оценка дисперсии ratio-метрики

In [ ]:
def var_ratio(x, y):
    """Delta-method оценка дисперсии отношения mean(x)/mean(y)."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    if len(x) < 2 or np.mean(y) == 0:
        return np.nan
    mean_x, mean_y = np.mean(x), np.mean(y)
    var_x, var_y = np.var(x, ddof=1), np.var(y, ddof=1)
    cov_xy = np.cov(x, y, ddof=1)[0, 1]
    return var_x / mean_y**2 + var_y * mean_x**2 / mean_y**4 - 2 * mean_x * cov_xy / mean_y**3

# Пример на пользовательско-дневных числителях/знаменателях.
# Для CTR используем clicks/views; результат имеет смысл только при наличии просмотров баннера.
ratio_rows = []
for metric, numerator, denominator in [
    ("CTR", "clicks", "views"),
    ("Avg Watch Time", "watch_time", "clicks"),
    ("CR_fav", "adds_to_fav", "clicks")
]:
    if numerator in funnel and denominator in funnel:
        ratio_rows.append({"metric": metric, "delta_method_variance": var_ratio(funnel[numerator], funnel[denominator])})
display(pd.DataFrame(ratio_rows))

## 5. MDE в зависимости от размера выборки

In [ ]:
ALPHA = 0.05
POWER = 0.80
EXPECTED_LIFT = 0.03
MAX_SHARE = 0.15
N_USERS = df["client_id"].nunique()
eligible_total = int(np.floor(N_USERS * MAX_SHARE))
# Для равного распределения между двумя группами 50/50.
n_per_group_cap = eligible_total // 2

def mde_absolute(std, n_per_group, alpha=ALPHA, power=POWER):
    """Двусторонний нормальный приближенный MDE для двух независимых равных групп."""
    if not np.isfinite(std) or n_per_group <= 0:
        return np.nan
    z_alpha = stats.norm.ppf(1 - alpha / 2)
    z_power = stats.norm.ppf(power)
    return (z_alpha + z_power) * std * np.sqrt(2 / n_per_group)

# Основная метрика: среднее время просмотра после клика.
metric = funnel["avg_watch_time"].dropna()
mu = metric.mean()
std = metric.std(ddof=1)
print(f"Среднее Avg Watch Time: {mu:.4f}")
print(f"Стандартное отклонение: {std:.4f}")
print(f"Всего уникальных пользователей в файле: {N_USERS}")
print(f"Максимум пользователей в эксперименте (15%): {eligible_total}")
print(f"Максимум на группу при распределении 50/50: {n_per_group_cap}")
if n_per_group_cap >= 2 and np.isfinite(std):
    absolute_mde = mde_absolute(std, n_per_group_cap)
    relative_mde = absolute_mde / mu * 100 if mu and np.isfinite(mu) else np.nan
    print(f"Оценочный MDE: {absolute_mde:.4f} минут; {relative_mde:.2f}% относительно среднего")
    print(f"Ожидаемый эффект 3% {'может быть обнаружен по грубой оценке' if relative_mde <= 3 else 'не обнаруживается при таком размере выборки по грубой оценке'}")
else:
    print("Недостаточно наблюдений для надёжной оценки MDE.")

In [ ]:
# График MDE от размера выборки (на группу), если дисперсия оценивается.
if np.isfinite(std) and std > 0:
    max_n = max(2, n_per_group_cap)
    sizes = np.unique(np.linspace(2, max_n, min(50, max_n-1), dtype=int))
    mdes = np.array([mde_absolute(std, n) for n in sizes])
    plt.figure(figsize=(8, 4.5))
    plt.plot(sizes, mdes / mu * 100 if mu else mdes, marker="o")
    plt.axhline(3, linestyle="--", label="Ожидаемый эффект 3%")
    plt.xlabel("Размер одной группы")
    plt.ylabel("Относительный MDE, %")
    plt.title("Зависимость MDE от размера выборки")
    plt.legend()
    plt.grid(True, alpha=.3)
    plt.tight_layout()
    plt.show()
else:
    print("График MDE недоступен: недостаточно валидных наблюдений или нулевая вариативность.")

## 6. Рекомендация по дизайну

- Рандомизировать пользователей, а не события, чтобы один пользователь не попадал в обе группы.
- Максимум — 15% всей доступной аудитории; распределить эту выборку поровну: 7,5% в контроль и 7,5% в тест.
- Основная метрика: среднее время просмотра после клика по баннеру.
- Вторичные метрики: CTR, конверсия в избранное после клика и watch time на пользователя.
- Продолжительность — ровно 7 дней; заранее зафиксировать критерии исключения, метод расчёта метрик, alpha=0.05 и мощность 80%.
- Не запускать тест, если расчёт мощности показывает, что при ограничении выборки 15% невозможно обнаружить ожидаемый прирост 3%. В таком случае согласовать увеличение срока/доступной аудитории либо признать тест недостаточно мощным.

**Ограничение:** 100 событий — очень маленькая историческая выборка, и она не позволяет надёжно оценить параметры для всей платформы. Для решения о запуске нужен более полный исторический период.